# 07 · Deposition and optical depth

This notebook goes from the optical-depth integral along a path to the sum over pieces that SkewerJAX evaluates, and
from the gas cells to $\tau$, on one skewer through the TNG50-4 snapshot of notebook 05, with the package's own code
path. The derivation comes first, below; then:

1. the deposition, step by step on that skewer: selection, deposition into pieces, the sum over the pieces, and
   periodic boundaries;
2. the line-centred frames, the global velocity axis and the observed wavelength, for $\mathrm{Ly}\alpha$ and Si III on
   the same skewer.

The profile that every piece adds, that of a gas element, is derived in notebook 06. The closed-form checks of $\tau$
(static gas with natural or thermal widths, radiation-dominated expansion) are in
`tests/core/test_optical_depth.py`. Runtime: ~1 min.

### From the integral to the sum over pieces

**The integral.** Along a straight comoving path, a photon is absorbed by transition $j$ of species $i$ with optical
depth (review notes, Eq. `tau_straight_los`, with $dl_{\rm phys} = a\, dx$)

$$
\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^3}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, a(x)\, dx
= \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^2}\, \phi_j\big(\nu_j(v_\dagger, x)\big)\, dx,
\qquad \mathcal{C}_j = \frac{e^2 f_j}{4\varepsilon_0 m_e c} .
$$

It is a line integral along the path of the light. At every $x$ the gas adds its line profile $\phi_j$: the Voigt
function at the detuning $\nu_j$ of the photon from the line centre $v_c(x)$ of the gas, with the widths $\sigma$ and
$\gamma$ of its temperature and transition (notebook 06, §2).

**Step 1: cut the path into pieces.** The integral is the sum of the integrals over the parts of any partition of the
path. SkewerJAX uses **pieces** $p$, of length $\ell_p$: fragments of the path within which the gas properties are
constant. Every source gives the gas as Voronoi cells (`core.Cells`: the toy models of 04, the snapshots of 05), uniform
inside each cell, so the pieces are the exact intersections of the path with the cells (§1). By default they are also
cut at the ends of the dual segments of the sampling points, one piece per cell and dual segment; the cell crossings
(`deposition="voronoi_crossings"`) keep one interval per crossing of a cell, split into equal sub-pieces.

**Step 2: what is constant within a piece, and what is not.** Constant, exactly, because a piece lies in one cell:
$n_{\rm com}$, $T$ (and so $\sigma$ and $\gamma$) and the velocity vector; the path is straight within a piece, so
$v_\parallel = \mathbf{v}\cdot\hat t$ is constant too. Not constant: $a(x)$, so $a^{-2}$ and the Hubble part of the line
centre, $v_H(x) = c\ln(a_\dagger/a(x))$, change across every piece. The line centre moves across a piece by

$$
W_p = \left\lvert\frac{dv_c}{dx}\right\rvert \ell_p = \left\lvert\frac{dv_H}{dx}\right\rvert \ell_p ,
$$

the Hubble rate times the length of the piece.

**Step 3: one profile per piece, at its centroid.** The scale factor $a$, $a^{-2}$ and $v_c$ are evaluated at the
centroid $s_p$ of every piece ($a$ interpolated in $\ln a$ between the bundle's points, where $a(x)$ was solved), with
the values of its cell. This is the midpoint rule on each piece:

$$
\tau_j(v_\dagger) \simeq \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\, \ell_p}{a_p^2}\, \phi_{j,p}(v_\dagger),
\qquad \phi_{j,p}(v_\dagger) = V\big(\nu_j(v_\dagger - v_{c,p});\, \sigma_p, \gamma_p\big),
$$

with the widths $\sigma_p$, $\gamma_p$ of the piece's cell and the exact detuning of notebook 06, §2. This is
`core.optical_depth_kernel`, mapped over skewers by `optical_depth_from_pieces`, behind
`compute_optical_depth(source, v, lines, ...)`, whose source is a `SkewerBundle` with its `Cells`, or `Pieces`.

**Error, and when the sum is exact.** Both follow from the gas being piecewise constant:

- The midpoint rule replaces the profile smeared over $W_p$ by the profile at its centre. For a Gaussian core this
  changes the peak by a relative $W_p^2/(12 b^2)$: the sum converges to the integral at second order in the piece
  length, i.e. in the spacing of the sampling points.
- $\phi$ has unit area in $\nu$, so $\int \tau\, d\nu = \mathcal{C}_j \sum_p n_{{\rm com},p}\, \ell_p\, a_p^{-2}$ for any
  partition. The column $\sum_p n_p \ell_p$ is exact, for any spacing, because $n$ is exactly constant in a cell and the
  lengths are exact.
- The sum equals the integral exactly when $a$ and $v_c$ are constant within every piece; only the Hubble flow varies
  inside a cell. It converges to the integral as $\max_p W_p/b_p \to 0$.
- The resolution check of `compute_optical_depth` (`resolution_report`) monitors exactly this ratio, with the width of
  every piece (`Pieces.width_kms`). With the simulation default of 0.25 km/s spacing, $W_p \le 0.25$ km/s, far below
  any thermal width.

§1 builds this sum step by step, on a skewer through TNG50-4, and adds the one option that changes where the pieces
absorb: a periodic spectrum.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
# float32, the package's precision; skewerjax.config.set_precision("float64") for float64 (docs/precision.md)
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style("dark")          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'simulation', 'fields', 'profiles', 'deposition', 'tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, SkewerBundle, compute_optical_depth, resolution_report, phys)
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")             # the two lines of this notebook
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")

## 1 · Deposition and periodic boundaries, step by step

Three different grids meet in a simulated spectrum, and the deposition is what connects them:

- **the cells:** one value per Voronoi cell, the region closer to its mesh-generating point than to any other;
- **the sampling points of the `SkewerBundle`:** point $i$ stands for its *dual segment*, between the midpoints with its
  neighbours, of length $w_i$ (03);
- **the spectral grid** of $v$ on which $\tau$ is evaluated, chosen freely.

What connects them are the **pieces**: the fragments of the path inside one cell, within which the gas is constant.
Every source of gas reaches $\tau$ as cells (`core.Cells`: the toy models of 04, the snapshots of 05), and from there
by one route. The steps below run that route, one function per step, on a real skewer; `compute_optical_depth(bundle,
v, lines, cells)` chains the same functions in one call, for a toy model or a snapshot alike. The last step makes the
spectrum periodic, an option of $\tau$.

**The skewer.** A straight line along $x$ through the centre of the TNG50-4 box ($y = z = L/2$), from one face to the
opposite one: a `SkewerBundle` with the snapshot's cosmology, its scale factor at the first point and the observer
before it. Parallel skewers across a box share one path coordinate, so `SkewerBundle.axis_aligned` (box mode) solves
$a(s)$ once, in float64 on the host, for all of them (notebooks 12, 13); this one is built from its points directly. Its points are 20 km/s of Hubble flow apart (`DV_POINTS`, 0.26 cMpc), about one per cell: this spacing is a
visualisation choice, so that the pieces of the figures below are few and large, and $\tau$ at it is not converged (the
resolution check of step 4 warns, and §2 uses the same line sampled every km/s). **The cells** are those of a slab of $\pm1.5$ cMpc around it, read as in
notebook 05 (`snapshot_cells`: the gas physics of 05, §2, per cell, for H I and Si III).

In [ ]:
from skewerjax.utils.paths import DATA_DIR as DATA      # folder with TNG50-4/: $SKEWERJAX_DATA (docs/installation.md)
from skewerjax.simulations import ArepoHDF5Reader, snapshot_cells
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
L, yc = H.box_cMpc, H.box_cMpc / 1.16
cosmo = H.cosmology()                                     # the snapshot's flat LCDM
DV_POINTS = 20.0                                          # [km/s] point spacing: coarse, for the figures (not converged; §2 uses 1 km/s)
x = np.linspace(0.0, L, int(round(H.vmax_kms / DV_POINTS)) + 1)           # first = last place
skewer = SkewerBundle(spatial_points=jnp.asarray(np.stack([x, np.full_like(x, yc), np.full_like(x, yc)], -1)[None]), a0=H.a, x0=0.0,
                      ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)   # observer before the first point
gas = snapshot_cells(reader, ["HI", "SiIII"], [0.0, yc - 1.5, yc - 1.5], [L, yc + 1.5, yc + 1.5])   # the slab: core.Cells, periodic in x
print(f"one skewer of {skewer.N_ppsk:,} points along x at y = z = {yc:.2f} cMpc, {float(skewer.path_lengths[0, -1]):.2f} cMpc; "
      f"a from {float(skewer.scale_factors[0, 0, 0]):.5f} to {float(skewer.scale_factors[0, 0, -1]):.5f}")
print(f"{len(gas):,} cells in the slab around it, out of {H.n_gas:,}")

**1. Selection.** `select_cells` keeps the cells within reach of the points, as in notebook 05 (the same rule as the
streamed `select_cells_near_paths` of `SimulationSkewers`).

In [ ]:
from skewerjax.core import (select_cells, voronoi_segments, cell_crossings, absorbers_from_components, bundle_frame, absorber_pieces,
                            cell_pieces)
near = select_cells(gas, skewer.spatial_points)                              # within 2.5 cell sizes of a point
print(f"selection: {len(near):,} of {len(gas):,} cells can reach the skewer")

**2. Deposition.** `voronoi_segments` cuts every dual segment at the exact boundaries of the cells it crosses: one
piece per (cell, dual segment). Along a straight path $\mathbf x(s) = \mathbf x_0 + s\,\hat t$, the squared distance to a
generator at $u_g = \hat t\cdot(\mathbf g - \mathbf x_0)$ along the path and $r_g$ from it is

$$
\lvert\mathbf x(s) - \mathbf g\rvert^2 = (s - u_g)^2 + r_g^2 = s^2 - 2u_g\,s + u_g^2 + r_g^2,
$$

and $s^2$ is common to all generators: the nearest one at every $s$ is the lowest of the straight lines
$-2u_g s + u_g^2 + r_g^2$. Their lower envelope (a convex-hull sweep over the generators sorted by $u_g$, $O(K\log K)$)
gives every cell the path crosses and its exact crossings, $s^* = \tfrac12(u_i + u_j) + (r_i^2 - r_j^2)/[2(u_i - u_j)]$,
which are then cut at the ends of the dual segments. A curved path is straight on every half of a dual segment: there
each half is sampled at sub-points labelled by a k-d tree, and every change of label is located at the crossing of the
same two lines. `cell_crossings` joins the pieces of one cell across the midpoints, which leaves one interval per
crossing of a cell.

**The crossings are the default pieces of a straight path.** Inside a crossing the gas is constant and only the line
centre moves, with the Hubble flow; `compute_optical_depth(bundle, v, lines, cells)` on straight paths uses the crossings
themselves (`core.crossings`: the same envelope, the candidates found by index arithmetic, no sampling points) and
integrates the profile over each one (step 4), so the sampling of the bundle no longer matters there. The dual-segment
pieces drawn here, cut again at the ends of the dual segments, are those of curved paths and of
`deposition="voronoi_segments"`; this section uses them to show both kinds of cut.

In [ ]:
pts = skewer.spatial_points
seg = voronoi_segments(pts, near.positions, L, gas_size=near.size_cMpc)        # pieces of every dual segment
cr = cell_crossings(seg, pts)                                                 # the same pieces joined per cell
print(f"deposition: {int((seg.cell >= 0).sum()):,} pieces in {skewer.N_ppsk:,} dual segments; {int(cr.n_crossings[0])} cell crossings")
print(f"the pieces tile the path: sum of their lengths {seg.length[seg.cell >= 0].sum():.6f} = path length "
      f"{float(skewer.path_lengths[0, -1]):.6f} cMpc")

**3. Pieces.** `absorbers_from_components` gives every piece its centroid $s_p$, its cell's density and
temperature, and the projection of its cell's velocity on the path, $v_{\parallel,p} = \mathbf{v}\cdot\hat t$. The
frame of the bundle (`bundle_frame`) gives $a(s_p)$ along the path, the line-centre span $W_p$ of the piece (the Hubble
flow across it) and the direction of the photons, and `absorber_pieces` turns them into `Pieces`, what the sum runs
over. `cell_pieces` runs the three steps for every species of a `LineBundle` in one call, from the cells of the slab;
on this straight path its default pieces are the cell crossings themselves (`deposition="voronoi"`), one per crossing,
and the figures below show those.

In [ ]:
lya = LineBundle(["HI-LyA"], ref_line_id="HI-LyA")
ab = absorbers_from_components(seg, pts, near, "HI", float(lya.mass_amu[0]))   # s (centroid), n, l, T, v_par of every dual-segment piece
frame = bundle_frame(skewer)                                                  # a(s) along the path, a_dagger = a(0), photon direction
pieces = absorber_pieces(ab, frame, lya.n_lines)                             # Pieces: what the tau sum runs over
same, parts = cell_pieces(gas, lya, skewer_bundle=skewer, return_parts=True)  # the default in one call: the cell crossings (and their parts)
column = lambda p: float(np.sum(np.asarray(p.number_density, np.float64) * np.asarray(p.length_m, np.float64)))   # sum n l
n_cr = int((np.asarray(same.length_m) > 0).sum())
print(f"cell_pieces (selection + crossings + pieces in one call): {n_cr} crossings for the {int((np.asarray(pieces.length_m) > 0).sum())} "
      f"dual-segment pieces; the same column to {abs(column(same) / column(pieces) - 1):.1e}")
live = ab.n[0] > 0
print(f"{int(live.sum()):,} dual-segment pieces in {len(np.unique(ab.cell[0][live]))} cells; v_par from {ab.v_los[0][live].min():+.0f} to "
      f"{ab.v_los[0][live].max():+.0f} km/s; Hubble widths W_p up to {float(pieces.width_kms.max()):.2f} km/s (crossings: "
      f"{float(same.width_kms.max()):.2f}); velocity span of the path {float(pieces.velocity_span_kms[0]):.1f} km/s")

**4. The optical depth.** `core.compute_optical_depth` sums one profile per piece,

$$
\tau(v) = \sum_p d\tau_p(v),\qquad d\tau_p(v) = \mathcal{C}_j\, n_{{\rm com},p}\, \ell_p\, a_p^{-2}\, \langle\phi_{j}\rangle_p(v),
$$

where $\langle\phi_j\rangle_p$ is the profile **averaged over the line centres the piece spans**: across a piece only the
line centre moves, from $v_c(s_{\rm lo})$ to $v_c(s_{\rm hi})$ (a width $W_p$ of Hubble flow), and the mean of Harris'
expansion over an interval has a closed form (`core.voigt_integral`), weighted by $a^{-2}\,ds/dv_c$ along the piece. So a
piece of any length is exact: no sampling, no resolution condition (`profile_integration="integrated"`, the default). The
option `profile_integration="midpoint"` evaluates the profile at the line centre of the centroid, $v_{c,p} \simeq
c\ln(a_\dagger/a(s_p)) + v_{\parallel,p}$, the earlier default: it needs pieces narrower than the line, $W_p \leq b/4$,
and the resolution check of `compute_optical_depth` warns otherwise. With a point every 20 km/s the pieces here are up
to 20 km/s wide, about twice the thermal width: the midpoint option warns and errs (below), the integrated profile
gives the result of the crossings. `per_piece=True` returns every $d\tau_p$.

**How the sum is computed** (`core.tau_engine`; `docs/performance.md`). Each piece is evaluated exactly on a window of
pixels around its line centre, sized so that what lies beyond is its Lorentz wing to $10^{-9}$ (`wing_tolerance`); the
wings beyond the windows, $\propto (v - v_c)^{-2}$ for every piece, are one convolution, added exactly by FFT. The Voigt
profile is Harris' expansion, $H(a, u) \simeq e^{-u^2} + a\,H_1(u)$ for $a \le 10^{-3}$ (fourth order in $a$ up to
$a = 0.05$, ExoJAX's exact function beyond; notebook 06). fake_spectra's per-cell schemes, `pixel_average` (a narrow
cell's profile averaged over the pixel) and `tautail` (each cell walks out from its centre pixel until it adds less
than $10^{-7}$), run per cell on a window of pixels around its line centre, widened until the walk stops inside
(`tau_engine.per_cell_line_tau`): the result of the whole grid at the cost of the window. Under `jax.jit`, with
`per_piece=True` or on a non-uniform grid the sum is dense (every piece on every pixel, the same integrated profile).
`devices=skewerjax.config.devices("cpu")` or `devices("gpu")` sends the sum to those devices, in chunks;
`skewerjax.config.configure_devices` sets the CPU devices before JAX starts.

In [ ]:
import warnings
P = float(pieces.velocity_span_kms[0])                                     # the velocity span of the path, c ln(a_first / a_last)
v = np.arange(-300.0, P + 300.0, 1.0)                                      # 1 km/s pixels over the path and 300 km/s beyond each end
tau = np.asarray(compute_optical_depth(pieces, v, lya))[0, 0]              # the default: each piece's profile integrated over it
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    tau_mid = np.asarray(compute_optical_depth(pieces, v, lya, profile_integration="midpoint"))[0, 0]   # the option: one point per piece
rep = resolution_report(pieces, lya)
print(f"tau = sum over {int(live.sum()):,} pieces: max {tau.max():.1f}, <F> on [0, P) = {np.exp(-tau[(v >= 0) & (v < P)]).mean():.3f}; "
      f"max W/b = {rep['max_ratio'][0]:.2f}, column in pieces wider than b/4: {rep['column_fraction'][0]:.0%}"
      f" (the midpoint option warns: {any('resolution' in str(x.message).lower() for x in w)})")
x1 = np.linspace(0.0, L, int(round(H.vmax_kms)) + 1)                         # the same line, one point per km/s (used in section 2)
skewer_fine = SkewerBundle(spatial_points=jnp.asarray(np.stack([x1, np.full_like(x1, yc), np.full_like(x1, yc)], -1)[None]), a0=H.a, x0=0.0,
                           ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
tau_cross = np.asarray(compute_optical_depth(skewer, v, lya, gas))[0, 0]     # bundle + cells: the cell crossings, no sampling points
F = lambda t: np.exp(-t.astype(np.float64))
print(f"the crossings of the same path (selection, crossings, contributions and tau in one call): max |dF| against the "
      f"integrated pieces {np.abs(F(tau_cross) - F(tau)).max():.1e}, against the midpoint option {np.abs(F(tau_cross) - F(tau_mid)).max():.2f}")

**5. Periodic boundaries.** The cells are periodic: the gas past a face of the box is the gas at the opposite face, and
the selection and the deposition use its nearest images. That is the topology of the gas, not an option. The spectrum
is not periodic by nature. The light that crosses the path once is absorbed by the Hubble flow from $v_H = 0$ at one end
to $v_H = P = c\ln(a_{\rm first}/a_{\rm last})$ at the other, so the two ends map to different velocities, and what falls
beyond them is not on the spectrum. A periodic spectrum is a choice, `velocity_period` of `compute_optical_depth`:

- `False` (the default): every piece absorbs where its line centre is;
- `True`: each skewer is periodic over its own velocity span $P$, set by the scale factors at its first and last points
  (`Pieces.velocity_span_kms`);
- a number: that period [km/s] for every skewer.

**How the code wraps: one operation, on the detuning.** For every piece $p$, the
detuning $v - v_{c,p}$ is replaced by its nearest image modulo $P$, in $[-P/2, P/2)$ (`spectral.frequency_offset_GHz`).
The line centres themselves are never moved, nothing is computed outside the window.
So each piece's whole profile, centre and tails together, is laid on a circle of circumference $P$: a profile
near an edge is split, the part on one side stays where it is and the part beyond the edge is evaluated $P$ away, at the
other end. Two consequences are visible (rows (d) and (e) of the figures below):

- **(i)** a piece whose line centre falls outside the window absorbs at the other
  end, at $v_c \mp P$, apart from whatever part of its profile still lies inside;
- **(ii)** a piece whose line centre is inside, but whose profile reaches past an edge, has that tail re-enter at the
  other end.

**The cut at $P/2$.** Each pixel sees only the nearest image of each piece: the profile is cut half a period from its
centre, on the ray opposite it on the ring (the whole rings of rows (d) and (e)), and the other images are not summed. The exact periodic spectrum is the sum over all images,
$\sum_n \tau_1(v_\dagger + nP)$; the difference matters only for damped wings in small boxes.

In [ ]:
from skewerjax.core.spectral import line_centre_velocity_kms
tau_wrap = np.asarray(compute_optical_depth(pieces, v, lya, velocity_period=True))[0, 0]   # on the grid only; outside [0, P) it repeats
inside = (v >= 0) & (v < P)
n_px = int(np.ceil(P)); dv_w = P / n_px
v_win = (np.arange(n_px) + 0.5) * dv_w                                       # pixels that tile the window [0, P) exactly
per_k = [np.asarray(compute_optical_depth(pieces, v_win + k * P, lya, per_piece=True, resolution_tolerance=None))[0, 0]
         for k in (-1, 0, 1)]                                                # every piece, no wrap, on the windows below, at and above [0, P)
column = sum(t.sum() for t in per_k) * dv_w                                  # the integral of tau dv over [-P, 2P): the whole column
t_in, t_out = per_k[1].sum(0), per_k[0].sum(0) + per_k[2].sum(0)             # no wrap: inside the window, and P below / above it
t_wrap = np.asarray(compute_optical_depth(pieces, v_win, lya, velocity_period=True, resolution_tolerance=None))[0, 0]
v_c = np.asarray(line_centre_velocity_kms(pieces.a_dagger[0], pieces.scale_factor[0], pieces.los_velocity[0]))[0]
centre_out = live & ((v_c < 0) | (v_c >= P))
tails_ii = (per_k[0] + per_k[2])[live & ~centre_out].sum() * dv_w           # (ii): tails of centre-inside pieces beyond an edge
cut = ((t_in + t_out) - t_wrap).sum() * dv_w                                 # all images summed minus the nearest image
print(f"P = {P:.1f} km/s. Without wrap the window holds {t_in.sum() * dv_w / column:.4f} of the column: "
      f"{per_k[2].sum() * dv_w / column:.2%} falls beyond P, {per_k[0].sum() * dv_w / column:.1e} below 0")
print(f"(i) {int(centre_out.sum())} pieces have their line centre outside the window ({(per_k[0] + per_k[1] + per_k[2])[centre_out].sum() * dv_w / column:.2%} "
      f"of the column); (ii) the tails of centre-inside pieces across an edge hold {tails_ii / column:.1e} of it")
print(f"with velocity_period=True the window holds {t_wrap.sum() * dv_w / column:.6f} of the column; the cut at P/2 (images not summed) "
      f"removes {cut / column:.1e}")
print(f"balance: the no-wrap tau outside the window, {t_out.sum() * dv_w:.2f} km/s, = the gain inside it, "
      f"{(t_wrap - t_in).sum() * dv_w:.2f} km/s, + the cut, {cut:.2e} km/s")

**The figures.** A crossing of the box passes through hundreds of cells, and periodicity acts where the path closes on
itself. So the figures show only the pieces around the seam $s = 0 \equiv L$: those whose centroids lie within
$l$ = `HALF_RANGE_CMPC` of it, the last and the first $l$ of the path, **the end of the box on the left and its start on
the right**, so that the seam is in the middle. Every cell keeps one colour, its H I density, and one label (S1, S2, ... at
the start; E1, E2, ... at the end). From row (b) on, the pieces are the objects: each is labelled by its cell and a letter
in path order (one crossing per cell, E3a; S1, crossed at both ends of the path, gives S1a and S1b), with the same colour
and label in every row. Three figures:

- **(a)** the cells in 3D; **(b)** the pieces along $s$;
- **(c)–(e)** the same pieces on rings, the natural geometry of a periodic coordinate, each window centred on its seam (12
  o'clock) with the whole ring on its left: (c) the ring of $s$ over $[-l, l]$, (d) and (e) the ring of $v_\dagger$ over the
  fixed window $\pm$`HALF_RANGE_KMS` around $v_\dagger = 0 \equiv P$.

Every figure sets the length of its velocity arrows from the velocities it draws (`velocity_arrow_scale`: the
90th-percentile speed is 0.8 median cell sizes long) and gives it with a reference arrow of a round speed: in the 3D
scene of each panel of (a) (lengths in projection), and in a box at the top of (b) and of (c), which draw the same
$v_\parallel$ at one scale. Every quantity comes from the objects
above (`parts` of `cell_pieces`, and every $d\tau_p$ of `compute_optical_depth(..., per_piece=True, velocity_period=True)`
on one period).

In [ ]:
from skewerjax.utils.plotting import plot_deposition_cells, plot_deposition_pieces, plot_deposition_rings
HALF_RANGE_CMPC = 1.5     # [cMpc] l: the figures show the pieces with centroids in s in [-l, l) around the seam s = 0 = L
HALF_RANGE_KMS = 160.0    # [km/s] window of (d), (e): v_dagger in [-300, 300] around 0 = P; holds every shown piece after its v_par shift
fig, axs, info = plot_deposition_cells(parts, skewer, half_range_cMpc=HALF_RANGE_CMPC)
fig.suptitle("The two ends of the skewer are one place: the path leaves the box at x = L and comes back at x = 0, in the same cell S1",
             fontsize=21, y=1.03)
plt.show()
print(f"end: {info['n_cells']['end']} cells, {info['n_pieces']['end']} pieces; start: {info['n_cells']['start']} cells, "
      f"{info['n_pieces']['start']} pieces; {sum(info['n_pieces'].values())} pieces in s in [-{HALF_RANGE_CMPC:g}, {HALF_RANGE_CMPC:g}) cMpc")

*Reading (a).* The cells, as in notebook 05, in the colour of their H I density, with their peculiar velocities
$\mathbf{v}$ (arrows from the generating points, in the colour of their cell), the skewer and its unit tangent $\hat t$ (red), and the box face
where the path ends, (a1) at $x = L$, and starts, (a2) at $x = 0$ (green planes). Cell S1 is crossed at both ends:
$x = 0 \equiv L$ is one place.

In [ ]:
fig, axs, info = plot_deposition_pieces(parts, skewer, half_range_cMpc=HALF_RANGE_CMPC)
fig.suptitle("The pieces around the seam: one piece per cell crossing, the gas moving away from the seam on both sides",
             fontsize=21)
plt.show()
lab = info["piece_labels"]                                                  # {"E3a": index of the piece in the absorbers}
v_par = np.asarray(parts["absorbers"]["HI"].v_los[0])[list(lab.values())]
print(f"v_par of the {len(lab)} pieces {v_par.min():+.0f} to {v_par.max():+.0f} km/s "
      f"({v_par.min() * L / P:.1f}-{v_par.max() * L / P:.1f} cMpc of Hubble flow)")

*Reading (b).* Each bar is one piece $p$, the crossing of one cell by the path, of height $n_p$ and width $\ell_p$;
the white lines are the cell boundaries, where the path passes from one cell to the next. The arrow of a piece is its
$v_\parallel = \mathbf{v}\cdot\hat t$, as `absorbers_from_components` computes it, drawn from the centre of the piece
along $s$, with its sign; arrows that would overlap are stacked. Here $v_\parallel$ runs from $-25$ to $+19$ km/s
(printed above): the gas at the end of the box moves to smaller $s$ and that at the start mostly to larger $s$, away from
the seam on both sides. S1b, the last piece at the end, and S1a, the first at the start, are one cell crossed across the
seam: its two pieces have one $n$, one $T$ and one $v_\parallel$.


In [ ]:
n_px = int(np.ceil(P / 0.5)); v_ring = (np.arange(n_px) + 0.5) * P / n_px   # one period [0, P), 0.5 km/s pixels
dtau_p = np.asarray(compute_optical_depth(same, v_ring, lya, per_piece=True, velocity_period=True,
                                          resolution_tolerance=None))[0, 0]   # every d tau_p (N_p, N_v), wrapped
fig, axs, info = plot_deposition_rings(parts, skewer, v_ring, dtau_p, period=P, half_range_cMpc=HALF_RANGE_CMPC,
                                       half_range_kms=HALF_RANGE_KMS)
fig.suptitle("On the rings the seam disappears: the pieces tile the ring of s, and their peculiar velocities turn them on the ring of v",
             fontsize=21, y=1.01)
plt.show()
lo, hi = info["v_c_range"]
print(f"line centres of the shown pieces (+- W_p/2): {lo:+.0f} to {hi:+.0f} km/s, inside +-HALF_RANGE_KMS = {HALF_RANGE_KMS:g} km/s: "
      f"{max(abs(lo), abs(hi)) <= HALF_RANGE_KMS}")
print(f"on the velocity seam: {info['seam_piece']}, v_c = P {info['seam_v_c']:+.1f} km/s, W_p = {info['seam_W']:.1f} km/s; "
      f"its profile is cut at v_c + P/2 = {info['seam_v_c'] + P / 2:.0f} km/s; all the cuts remove {cut / column:.0e} of the column")

*Reading (c)–(e).* The pieces of (b), with their colours and labels, on the rings; left, the whole ring, with the
window in grey.

- **(c) The ring of $s$.** A crossing of the box closes on itself, so $s$ is periodic over $L$. The bars of (b) become
  blocks on a ring: angle $s$, angular width $\ell_p$, height $n_p$, and $v_\parallel$ along the ring from the centre of
  each block (the scale of (b)). In space the pieces tile the ring and never overlap: S1a continues S1b across the seam.
- **(d) The ring of $v_\dagger$.** Each piece moves to its line centre $v_{c,p} = v_H + v_\parallel$, keeping its height
  and label, and its angular width becomes $W_p$, the span of its line centres. The Hubble flow alone maps $\pm 1.5$ cMpc
  to about $\pm 115$ km/s; the peculiar velocities here ($-25$ to $+19$ km/s) turn each piece by less than its neighbours'
  spacing, so the order of (c) is kept and the line centres span $-138$ to $+130$ km/s (inside `HALF_RANGE_KMS`, printed
  above). The gap at the seam closes: S1b and S1a, one cell, meet there. S1a (bold) lies on the seam: its line centre is
  1.8 km/s past $0 \equiv P$ and its span, $W_p = 13.4$ km/s, straddles it.
- **(e) From pieces to $\tau$.** Each block becomes its $d\tau_p(v_\dagger)$: a Voigt profile at the piece's temperature,
  integrated over the piece's line centres, of area $\propto n_p \ell_p a_p^{-2}$, with the detuning wrapped to
  $[-P/2, P/2)$. On the ring the wrap is seamless: S1a's profile runs on across $0 \equiv P$ as one curve. Black: the
  total $\tau$, the sum of all the pieces of the skewer, including those beyond $\pm l$. The radius grows as $\log\tau$
  from $\tau = 10^{-3}$, so that the weak profiles stay visible.
- **The cut at $P/2$** (dashed, on the whole rings of (d) and (e)): each profile is evaluated within half a period of its
  centre only. For S1a that is the ray opposite the seam, $v_c + P/2$; the images beyond it are not summed (printed
  above: all the cuts together remove 5e-7 of the column).


**When to use which.**

- **No wrap** (the default) is the integral along the path as it is: right for any path that is not a single crossing
  of a periodic box (a light cone, a curved path, a segment of a longer sightline), and the cheapest. For a box it
  leaves out the absorption that falls beyond the ends: in 16 TNG50-4 skewers the mean flux is higher by about
  $7\times10^{-3}$ than with a periodic spectrum, and the P1D differs by up to 25 % at $k < 0.1$ s/km.
- **`velocity_period=True`** gives a spectrum periodic over the box, on a grid of one period. FFT-based estimators of a
  box need it (09). Its approximations: wrapped gas keeps its own scale factor (that of the face it belongs to), and
  only the nearest image of each piece is kept, which matters for damped wings in small boxes.

**What happens at $P/2$.** With `velocity_period`, the detuning of every piece is taken to its nearest image,

$$
\delta_p(v) = v - v_{c,p} - P \left\lfloor \frac{v - v_{c,p}}{P} + \frac{1}{2} \right\rfloor \in [-P/2,\, P/2),
$$

so each pixel sees each piece once, through the image of its line centre nearest to it. On the ring, $\delta_p = \pm P/2$
is the point opposite the piece's line centre, and its profile is cut there: the part farther than $P/2$ from the centre
would belong to the next image, $v_{c,p} \pm P$, which is not summed. The exact periodic $\tau$ is the sum over all
images, $\sum_n d\tau_p(v + nP)$; the nearest image keeps one of them, and the others are the parts of the profile
beyond $P/2$. A Gaussian core is negligible there; only broad, damped (Lorentzian, $\propto (v - v_c)^{-2}$) wings reach that
far, so the cut matters only for very strong absorbers in small boxes. On this skewer it removes $1.3\times10^{-6}$ of the
column (printed in step 5). In the ring figure, the cut is the dashed ray of the whole rings of (d) and (e), opposite
the piece on the velocity seam (printed under the figure).

## 2 · Line-centred frames, the global velocity axis and the observed wavelength

These are three axes for the same spectrum; changing from one to another is a change of axis only.

- **The line-centred frame** of every line: each line $j$ is integrated on a grid of $v_\dagger$, its own frame. By notebook
  06, §2, the line centre of the gas at $x$ is the same $v_c(x)$ in every frame: it does not depend on $\lambda_j$
  (`core.spectral.line_centre_velocity_kms`).
- **The global velocity axis** of a reference line (here $\mathrm{Ly}\alpha$): the same observed wavelength sits at
  $v_{\rm Ly\alpha} = v_\dagger - \Delta v_j$ in the reference frame, with

$$
\Delta v_j = c\ln\frac{\lambda_{\rm Ly\alpha}}{\lambda_j}
$$

  (`LineBundle.delta_v`). Going to this axis is a relabelling only: line $j$, computed on its own window $[0, P)$, moves
  to $[-\Delta v_j, P - \Delta v_j)$, and the reference line stays where it is. `compute_optical_depth(...,
  line_frame="reference")` evaluates every line directly on that axis, line $j$ at $v + \Delta v_j$ in its own frame
  (`spectral.velocities_in_line_frames`), over whatever grid it is given: on a grid that holds both lines it would also
  evaluate $\mathrm{Ly}\alpha$ below 0, outside its window (the far wings of the absorbers near the start), so here
  each line is computed on its own window only.
- **The observed wavelength** (`spectral.line_wavelength_grid_A`):

$$
\lambda_{\rm obs} = \lambda_j\, \frac{a_{\rm obs}}{a_\dagger}\, e^{v_\dagger/c} .
$$

**The gas.** The skewer of §1, sampled every km/s (`skewer_fine`), for $\mathrm{Ly}\alpha$ and Si III $\lambda$1206, with a
periodic spectrum (`velocity_period=True`), as a box crossing calls for (§1). Si III lives only in dense, warm gas
around haloes (notebook 05, §2): along this line through the diffuse gas of the box centre its $\tau$ stays below
$10^{-8}$. **For display only, the Si III optical depth is multiplied by a factor `SI_BOOST`** (a power of 10 that brings
its strongest feature to $\tau \approx 1$, $10^8$ here), $F_{\rm SiIII} = e^{-{\rm SI\_BOOST}\,\tau_{\rm SiIII}}$, in all
the figures and labelled in the legends; the computation is unchanged, and no combined flux is drawn.

In [ ]:
from skewerjax.utils.plotting import plot_frames_axes
from skewerjax.core import wrap_to_ref_line
a_dag = float(skewer_fine.a0[0])                                              # the reference scale factor: a at the first point
v_box = np.arange(0.5, P, 1.0)                                               # each line's own window [0, P), 1 km/s pixels
tau_own = np.asarray(compute_optical_depth(skewer_fine, v_box, lines, gas, velocity_period=True, resolution_tolerance=None))[:, 0]
F_own = np.exp(-tau_own.astype(np.float64))                                 # F in float64: in float32 a weak line's exp(-tau) rounds to 1
dV = float(lines.delta_v[1])
peaks = []                                                                    # the three strongest Si III features, 150 km/s apart
for j in np.argsort(tau_own[1])[::-1]:
    if all(abs(v_box[j] - q) > 150 for q in peaks):
        peaks.append(v_box[j])
    if len(peaks) == 3:
        break
SI_BOOST = 15.0 ** np.round(np.log10(1.0 / tau_own[1].max()))                # display only: Si III's strongest feature at tau ~ 1
n_w = int(np.ceil(P)); v_one = (np.arange(n_w) + 0.5) * P / n_w           # (d): a uniform grid of exactly one period
tau_one = np.asarray(compute_optical_depth(skewer_fine, v_one, lines, gas, velocity_period=True, resolution_tolerance=None))
tau_to_ref = np.asarray(wrap_to_ref_line(tau_one, v_one, lines, velocity_period=P))   # every line on the Lya axis, modulo P
figs = plot_frames_axes(v_box, F_own, lines, a_dagger=a_dag, a_obs=1.0, tau_scale={"SiIII-1206": SI_BOOST},   # one figure per case,
                        wrapped=dict(v=v_one, tau=tau_one[:, 0], tau_ref=tau_to_ref[:, 0], period=P))   # all of the same size
titles = {"own": "In its own frame, each line absorbs at the same line-centred velocity for the same gas",
          "global": r"On the global $\mathrm{Ly}\alpha$ axis, Si III moves by $-\Delta v_j$ = " f"{-dV:+.0f} km/s and keeps its own window",
          "wavelength": r"In observed wavelength the two lines overlap: Si III falls into the $\mathrm{Ly}\alpha$ forest",
          "wrapped": r"wrap_to_ref_line: Si III on the $\mathrm{Ly}\alpha$ frame, turned by $-\Delta v_j$ modulo $P$"}
for letter, (key, (fig, ax)) in zip("abcd", figs.items()):
    add_panel_letters([ax], labels=[letter])
    fig.suptitle(titles[key], fontsize=21)
plt.show()
print(f"<F>: Lya {F_own[0].mean():.3f}, Si III 1 - {-np.expm1(-tau_own[1].astype(np.float64)).mean():.1e} (true tau, max {tau_own[1].max():.1e}); "
      f"drawn with tau x {SI_BOOST:.0e}: max {SI_BOOST * tau_own[1].max():.2f}")
print(f"global axis: Lya on [{v_box[0]:.1f}, {v_box[-1]:.1f}] km/s (its own window), Si III on [{v_box[0] - dV:.1f}, {v_box[-1] - dV:.1f}] km/s")
print(f"Delta v(Si III) = c ln(lambda_Lya / lambda_SiIII) = c ln({lines.lines[0].lambda0_A:.4f} / {lines.lines[1].lambda0_A:.4f}) = "
      f"{phys.c_kms * np.log(lines.lines[0].lambda0_A / lines.lines[1].lambda0_A):+.1f} km/s: the strongest Si III feature, at "
      f"{peaks[0]:.1f} km/s in its own frame, sits at {peaks[0] - dV:.1f} km/s on the Lya axis (bluer: shorter wavelength)")
si_ref = np.asarray(compute_optical_depth(skewer_fine, v_box - dV, LineBundle(["SiIII-1206"], ref_line_id="HI-LyA"), gas.select_species("SiIII"),
                                          line_frame="reference", velocity_period=True, resolution_tolerance=None))[0, 0]
print(f"line_frame='reference' on the relabelled grid gives the same Si III: max |dtau| / max tau = "
      f"{np.abs(si_ref - tau_own[1]).max() / tau_own[1].max():.1e}")
tau_ref_frame = np.asarray(compute_optical_depth(skewer_fine, v_one, lines, gas, line_frame="reference", velocity_period=True,
                                                 resolution_tolerance=None))
print(f"(d) wrap_to_ref_line (interpolated) vs line_frame='reference' (evaluated) on one period: Si III max |dtau| / max tau = "
      f"{np.abs(tau_to_ref[1, 0] - tau_ref_frame[1, 0]).max() / tau_ref_frame[1, 0].max():.1e}; "
      f"Lya unchanged: {np.array_equal(tau_to_ref[0, 0], tau_one[0, 0])}")


*Reading the figures.*

- (a) In its own frame each line absorbs where the gas is: the same gas at the same $v_\dagger$ in both lines. Si III
  (its $\tau$ multiplied by `SI_BOOST` for display) absorbs only where the gas is densest, under saturated
  $\mathrm{Ly}\alpha$ absorbers.
- (b) On the $\mathrm{Ly}\alpha$ axis, each line keeps exactly the window it was computed on (dash-dotted edges, in its
  colour). $\mathrm{Ly}\alpha$ stays on $[0, P)$; Si III moves by $-\Delta v_j = -2270$ km/s, to the blue, because its
  rest wavelength is shorter: the same gas absorbs Si III at a shorter observed wavelength, 2270 km/s bluer on the
  $\mathrm{Ly}\alpha$ axis.
- (c) On the observed-wavelength axis the two lines are $9.2\ \text{Å} \times (a_{\rm obs}/a_\dagger)$ apart, and they overlap:
  a Si III feature falls into the $\mathrm{Ly}\alpha$ forest of gas 2270 km/s closer to the observer.
- (d) `wrap_to_ref_line` does (b) after the computation: $\tau_j^{\rm ref}(v) = \tau_j(v + \Delta v_j)$ by linear
  interpolation, and for a periodic spectrum (`velocity_period=P`, on a uniform grid of exactly one period) modulo $P$.
  So Si III does not leave the window as in (b): it turns by $-\Delta v_j$ around the ring of §1 and stays on $[0, P)$,
  dashed (own frame) to solid (the $\mathrm{Ly}\alpha$ frame), as `line_frame="reference"` would evaluate it (printed above).